# Agent 第 15 课：Knowledge Bases for Amazon Bedrock

Phase 1 第 5 课我们手写过长期记忆 / RAG：embedding → vector store → retrieve → stuff into prompt。

这节课看 AWS 把这一整套**托管**成什么样：**Knowledge Base（KB）**。

学完这节你能：
1. 理解 KB = 数据源 + 向量库 + ingest pipeline + 查询 API 的打包服务
2. 知道"把 KB 挂到 Agent 上"和"直接 query KB"两种用法的区别
3. 判断什么时候用 KB、什么时候自己撸（OpenSearch / pgvector / 别的）

## 1. KB 的组成

一个 Knowledge Base = 下面 4 层：

```
Data Source (S3 bucket 里的 PDF/Markdown/TXT/DOCX …)
      │ 定时/手动同步
      ▼
Ingestion Pipeline
 ├── parsing（Amazon Textract / BedrockFoundationModelParser 做多模态）
 ├── chunking（Fixed / Semantic / Hierarchical / Custom）
 └── embedding（Titan v2 / Cohere Embed v3 / ...）
      │
      ▼
Vector Store
 ├── Amazon OpenSearch Serverless (默认推荐)
 ├── Aurora PostgreSQL + pgvector
 ├── Pinecone / Redis Enterprise / MongoDB Atlas
 └── Neptune Analytics (GraphRAG 场景)
      │
      ▼
查询 API
 ├── Retrieve              —— 只返回 chunks
 └── RetrieveAndGenerate   —— 返回合成回答（内部起一次 LLM 调用）
```

**对照 Phase 1**：chunking = 你手写的 `split_text`，embedding = 你调 Titan/cohere，vector store = 你 pickle 的 numpy 数组，Retrieve = 你的 cosine_topk 函数。KB 就是把这四样全包。

## 2. 两种使用模式

**模式 A：直接 query KB**（不涉及 agent）
```python
runtime.retrieve(knowledgeBaseId=KB, retrievalQuery={"text":"..."})
runtime.retrieve_and_generate(input={"text":"..."}, retrieveAndGenerateConfiguration={...})
```
适合：做一个普通 RAG 问答接口，不需要工具调用。

**模式 B：把 KB 关联到 Agent**
```python
agent_client.associate_agent_knowledge_base(agentId=..., knowledgeBaseId=..., description=...)
```
之后 Agent 的 orchestration 循环会在"需要查资料"时自动调 KB（Bedrock 内部管）。你不用写 retrieve 代码。

**什么时候用哪种**：
- 纯 Q&A，没有副作用动作 → 模式 A（轻量）
- Agent 已经要做 action + 还需要内部文档知识 → 模式 B
- 两种可以混用：Agent 挂 KB + 你也可以在别处直接 retrieve

In [ ]:
import boto3, os, uuid, time, json
os.environ.setdefault("AWS_REGION", "us-west-2")

s3 = boto3.client("s3")
agent_client = boto3.client("bedrock-agent")
runtime = boto3.client("bedrock-agent-runtime")
sts = boto3.client("sts")
ACCOUNT = sts.get_caller_identity()["Account"]
print("account =", ACCOUNT)

## 3. 最小 KB：从 S3 文档到可查询

**前置**：
1. S3 bucket，放几个 markdown / pdf / txt
2. Bedrock KB execution role，权限：读那个 bucket + 调 embedding model + 写 OpenSearch Serverless collection
3. 如果用 OpenSearch Serverless：一个 collection + data access policy 允许该 role 写入

> 这些 resource 第一次建比较繁琐。**教学建议**：先在 Console 点一遍（Bedrock Console → Knowledge bases → Create → 选 "Quick create new vector store"，AWS 会帮你自动建 OpenSearch collection + role）。
>
> 下面代码假设 KB_ID 已存在（Console 里建好再回来跑）。

In [ ]:
# 手动填这两个：
KB_ID = "XXXXXXXXXX"                 # Console 建完 KB 后复制
DATA_SOURCE_ID = "YYYYYYYYYY"        # KB 下的 data source id
S3_BUCKET = f"my-kb-docs-{ACCOUNT}"  # 你的文档 bucket

# 上传一个示例文档
doc = '''# 公司休假政策（示例）
员工每年享有 15 天带薪年假。
- 年假需提前 3 天在 HR 系统申请。
- 未使用的年假最多可累计至次年 6 月。
- 病假凭医生证明另算，不占年假额度。'''
s3.put_object(Bucket=S3_BUCKET, Key="policies/leave.md", Body=doc.encode())
print("uploaded")

In [ ]:
# 触发一次 ingestion（增量 sync）
job = agent_client.start_ingestion_job(
    knowledgeBaseId=KB_ID, dataSourceId=DATA_SOURCE_ID,
)
job_id = job["ingestionJob"]["ingestionJobId"]
print("ingestion job:", job_id)

while True:
    st = agent_client.get_ingestion_job(
        knowledgeBaseId=KB_ID, dataSourceId=DATA_SOURCE_ID, ingestionJobId=job_id,
    )["ingestionJob"]
    print("  status:", st["status"], "stats:", st.get("statistics"))
    if st["status"] in ("COMPLETE", "FAILED", "STOPPED"): break
    time.sleep(5)

## 4. Retrieve vs RetrieveAndGenerate

In [ ]:
# Retrieve：只拿 chunks，自己拼 prompt 自己生成
r = runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={"text": "员工年假能累计到下一年吗？"},
    retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
)
for i, res in enumerate(r["retrievalResults"]):
    print(f"--- chunk {i}  score={res['score']:.3f} ---")
    print(res["content"]["text"][:200])

In [ ]:
# RetrieveAndGenerate：Bedrock 内部做一次 LLM 合成
REGION = boto3.session.Session().region_name or "us-west-2"
MODEL_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/anthropic.claude-sonnet-4-6-20260101-v1:0"

rg = runtime.retrieve_and_generate(
    input={"text": "员工年假能累计到下一年吗？"},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": KB_ID,
            "modelArn": MODEL_ARN,
        },
    },
)
print(rg["output"]["text"])
print("\n--- citations ---")
for c in rg.get("citations", []):
    for ref in c.get("retrievedReferences", []):
        print("src:", ref.get("location"), "score:", ref.get("metadata"))

## 5. Chunking 策略：最关键的调参旋钮

KB 检索质量 70% 取决于 chunking。Bedrock 提供几种：

| 策略 | 说明 | 适合 |
|---|---|---|
| **Fixed-size** | 每段固定 token（默认 300，overlap 20%） | 通用文档 |
| **Semantic** | 用 embedding 相似度判段落边界 | 散文 / 论述文 |
| **Hierarchical** | 父 chunk + 子 chunk，检索小块回传大块 | 长文档 / 法律 / 学术 |
| **Custom** | Lambda 自定义 | 有特殊结构（JSON、代码、表格） |

**调参建议**：
- 开始用 Fixed (300 tokens, 20% overlap)，跑一圈 eval
- 召回率低就换 Semantic
- 回答缺上下文就换 Hierarchical（父子结构）
- 文档全是表格/代码 → Custom Lambda

**parsing** 同样关键：含大量表格/图片的 PDF → 打开 `BedrockFoundationModelParser`（内部用多模态模型解析），收费但效果质变。

## 6. 把 KB 挂给 Agent

In [ ]:
# 假设你已经有个 AGENT_ID（上一课建的那种）
AGENT_ID = "ZZZZZZZZZZ"

agent_client.associate_agent_knowledge_base(
    agentId=AGENT_ID, agentVersion="DRAFT",
    knowledgeBaseId=KB_ID,
    description="公司内部政策文档——年假 / 病假 / 报销 等。",
    knowledgeBaseState="ENABLED",
)
agent_client.prepare_agent(agentId=AGENT_ID)
print("associated KB to agent")

# 之后 invoke_agent 里问 "员工年假能累计吗" agent 会自动走 KB
# trace 里 orchestrationTrace.invocationInput.knowledgeBaseLookupInput 能看到查询 embedding

## 7. 什么时候别用 KB，自己撸

KB 省心，但有时候不合适：

- **检索延迟敏感**：KB retrieve 通常 300-800ms（OpenSearch serverless 冷启动能到 2s+）。你的应用要 < 100ms 就自建 Redis/pgvector。
- **特殊索引需求**：需要 HNSW 参数调优、按元数据做复合过滤、BM25 + 向量混合排序——KB 的可配置度不够，自建 OpenSearch 直接写 DSL。
- **数据量极大（> 100M chunks）**：KB 的默认 OpenSearch collection 成本会暴涨；自建能选更便宜的介质。
- **数据频繁变动**：KB 的 ingestion 最小粒度是 "一次 sync job"，无法做"改一条更新一条"。真要实时 upsert 就自己搓。
- **需要 re-rank 之外的后处理**（query expansion、multi-query、self-RAG 等策略） → 手写更灵活。

**甜蜜点**：1M 以内 chunks、接受秒级新鲜度、企业内部文档问答 / agent 辅助 → KB 直接上。

## 8. 成本 / 观测几条

- KB 的费用由三部分组成：**embedding 调用**（每条 chunk）+ **OpenSearch OCU 小时费**（collection 常驻）+ **retrieve/retrieveAndGenerate 的 LLM token**
- OpenSearch Serverless 最小 2 OCU（索引 1 + 查询 1）≈ 每月 \$350+——**小 demo 不要开**，用 Pinecone free tier 或 Aurora pgvector 更省
- 所有 retrieve 都能开 CloudWatch metrics：`RetrievalCount` / `RetrievalLatency` / `IngestionFailureCount`；**必须加 alarm**
- Trace 里能看到 Agent 到底把 KB 的哪些 chunks 塞进了 prompt——答非所问时先看这里

---

## 9. 工程直觉

1. **先 Console 点一个 KB 跑通**，再去脚本化。第一次上来就写 boto3 建 KB，IAM + OpenSearch policy 能卡你一整天。
2. **chunking 策略是真正的调参目标**，embedding 模型换来换去差距远没有 chunking 大。
3. **RetrieveAndGenerate 的 citation 字段要暴露给用户**——这是企业场景"可溯源"的关键。
4. **KB 不是万能 RAG**。延迟敏感 / 索引定制 / 高频更新时老老实实自建。
5. Phase 2 到这里你已经会：框架全景 + Converse tool use + 托管 Agent + Action Groups + KB。

---

## 下一批（Phase 2 后续）预告

还剩这些准备讲（按需排）：
- **16 Guardrails for Bedrock**：把第 10 课的安全层接到托管服务
- **17 Agent Memory / Session storage**：跨对话记忆、用户长期偏好
- **18 观测：CloudWatch + X-Ray + OpenTelemetry**
- **19 Multi-agent collaboration on Bedrock**（supervisor agent）
- **20 Cost & performance 调优清单 → 生产上线 checklist**

先把 11-15 消化了，下一批再开工。